In [0]:
%pip install openpyxl

"""
================================================================================
 Synthetic Clinical Trial Data Generator  (SDTM-aligned)
================================================================================
Generates realistic, referentially-consistent clinical trial subject-level data
for an end-to-end Pharma Data Engineering platform demo.

Core domains produced (10):
    DM  Demographics            (master subject registry)
    SV  Subject Visits          (visit actuals per subject)
    VS  Vital Signs
    LB  Laboratory
    EG  ECG
    EC  Exposure as Collected   (study drug administration)
    CM  Concomitant Medications
    AE  Adverse Events
    MH  Medical History
    DS  Disposition             (subject end-of-study status)

Plus trial-design reference (one row set per study): TA, TS (small, for joins).

Referential integrity:
    * DM is the parent. Every subject has a unique USUBJID = STUDYID-SITEID-SUBJID
    * All event/finding domains reference USUBJID that exists in DM
    * Visit-based domains reference VISITNUM/VISIT defined by the study visit schedule
    * Dates are internally consistent (consent <= screening <= treatment <= events)

Realistic business scenarios injected:
    * Screen failures (subject never randomized; only screening data)
    * Early dropouts (disposition = withdrawn; truncated visits)
    * Adverse events incl. serious AEs
    * Lab abnormalities (values outside reference range, flagged)
    * Missing visits (gaps in SV)
    * Protocol deviations (flag column)
    * Multiple visits per subject

Output formats (to exercise multi-source ingestion):
    DM, SV, VS, LB, EG  -> CSV
    EC, CM              -> CSV   (also represent SQL-Server / Oracle extracts)
    AE                  -> JSON  (represents an API / JSON feed)
    MH                  -> Excel (.xlsx)
    DS                  -> CSV
    TA, TS              -> CSV   (trial design reference)

Scale is configurable below. Defaults create a meaningful dataset quickly;
raise N_SUBJECTS_PER_STUDY to reach "thousands of subjects".
================================================================================
"""

import os
import csv
import json
import random
import datetime as dt
from pathlib import Path

import pandas as pd

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
SEED = 20260609
random.seed(SEED)

OUTPUT_DIR = Path("/Workspace/Users/barshilekiran14@gmail.com/IntelliBi_Databricks/IntellibiDatabrickspro/Data_factory_data/data/raw")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Studies, sites, countries
STUDIES = ["CT-ONCO-001", "CT-CARD-002"]
COUNTRIES = ["USA", "India", "Germany", "Japan", "Brazil"]
SITES_PER_COUNTRY = 2            # sites per country per study
N_SUBJECTS_PER_STUDY = 600       # raise to 2000+ for "thousands" total
SCREEN_FAIL_RATE = 0.12
DROPOUT_RATE = 0.18
PROTOCOL_DEVIATION_RATE = 0.08
SERIOUS_AE_RATE = 0.15           # of subjects with at least one AE

# Visit schedule (VISITNUM, VISIT, nominal day offset from first dose)
VISIT_SCHEDULE = [
    (1, "Screening", -14),
    (2, "Baseline / Day 1", 0),
    (3, "Week 2", 14),
    (4, "Week 4", 28),
    (5, "Week 8", 56),
    (6, "Week 12", 84),
    (7, "End of Treatment", 112),
    (8, "Follow-up", 140),
]

ARMS = [("Treatment A", "ARMA"), ("Treatment B", "ARMB"), ("Placebo", "PBO")]

SEX = ["M", "F"]
RACE = ["WHITE", "ASIAN", "BLACK OR AFRICAN AMERICAN",
        "AMERICAN INDIAN OR ALASKA NATIVE", "OTHER"]
ETHNIC = ["HISPANIC OR LATINO", "NOT HISPANIC OR LATINO"]

# Lab tests: code, name, unit, normal low, normal high, category
LAB_TESTS = [
    ("HGB", "Hemoglobin", "g/dL", 12.0, 17.5, "HEMATOLOGY"),
    ("WBC", "Leukocytes", "10^9/L", 4.0, 11.0, "HEMATOLOGY"),
    ("PLAT", "Platelets", "10^9/L", 150, 400, "HEMATOLOGY"),
    ("ALT", "Alanine Aminotransferase", "U/L", 7, 56, "CHEMISTRY"),
    ("AST", "Aspartate Aminotransferase", "U/L", 10, 40, "CHEMISTRY"),
    ("CREAT", "Creatinine", "mg/dL", 0.6, 1.3, "CHEMISTRY"),
    ("GLUC", "Glucose", "mg/dL", 70, 110, "CHEMISTRY"),
    ("K", "Potassium", "mmol/L", 3.5, 5.1, "CHEMISTRY"),
]

VS_TESTS = [
    ("SYSBP", "Systolic Blood Pressure", "mmHg", 100, 140),
    ("DIABP", "Diastolic Blood Pressure", "mmHg", 60, 90),
    ("PULSE", "Pulse Rate", "beats/min", 60, 100),
    ("TEMP", "Temperature", "C", 36.1, 37.5),
    ("RESP", "Respiratory Rate", "breaths/min", 12, 20),
    ("WEIGHT", "Weight", "kg", 50, 110),
]

EG_TESTS = [
    ("QT", "QT Interval", "msec", 350, 450),
    ("QTCF", "QTcF Interval", "msec", 350, 450),
    ("HR", "Heart Rate", "beats/min", 60, 100),
    ("PR", "PR Interval", "msec", 120, 200),
]

AE_TERMS = [
    ("Headache", "Nervous system disorders"),
    ("Nausea", "Gastrointestinal disorders"),
    ("Fatigue", "General disorders"),
    ("Diarrhoea", "Gastrointestinal disorders"),
    ("Dizziness", "Nervous system disorders"),
    ("Hypertension", "Vascular disorders"),
    ("Rash", "Skin and subcutaneous tissue disorders"),
    ("Neutropenia", "Blood and lymphatic system disorders"),
    ("Vomiting", "Gastrointestinal disorders"),
    ("Pyrexia", "General disorders"),
    ("Anaemia", "Blood and lymphatic system disorders"),
    ("Arthralgia", "Musculoskeletal disorders"),
]

CM_DRUGS = ["Paracetamol", "Ibuprofen", "Omeprazole", "Metformin",
            "Atorvastatin", "Amoxicillin", "Aspirin", "Loratadine"]

MH_TERMS = ["Hypertension", "Type 2 Diabetes Mellitus", "Asthma",
            "Hypercholesterolaemia", "Osteoarthritis", "Migraine",
            "Gastro-oesophageal reflux", "Depression"]

STUDY_DRUGS = {"CT-ONCO-001": "Oncozumab", "CT-CARD-002": "Cardiostat"}


def d(date_obj):
    """ISO date string (SDTM --DTC style, date only)."""
    return date_obj.strftime("%Y-%m-%d")


def rnd_around(low, high, allow_abnormal=0.0):
    """Random value within [low, high]; with prob allow_abnormal, push outside."""
    val = random.uniform(low, high)
    if random.random() < allow_abnormal:
        if random.random() < 0.5:
            val = low - random.uniform(0.05, 0.4) * (high - low)
        else:
            val = high + random.uniform(0.05, 0.5) * (high - low)
    return round(val, 1)


def build():
    dm_rows, sv_rows, vs_rows, lb_rows, eg_rows = [], [], [], [], []
    ec_rows, cm_rows, ae_rows, mh_rows, ds_rows = [], [], [], [], []
    ta_rows, ts_rows = [], []

    # Trial design reference (TA = arms, TS = summary) -- one set per study
    for study in STUDIES:
        for arm_name, arm_cd in ARMS:
            ta_rows.append({
                "STUDYID": study, "DOMAIN": "TA", "ARMCD": arm_cd,
                "ARM": arm_name, "TAETORD": 1, "ETCD": "TRT",
                "ELEMENT": "Treatment"})
        ts_rows.append({"STUDYID": study, "DOMAIN": "TS", "TSPARMCD": "TITLE",
                        "TSPARM": "Trial Title",
                        "TSVAL": f"A Phase III Study of {STUDY_DRUGS[study]}"})
        ts_rows.append({"STUDYID": study, "DOMAIN": "TS", "TSPARMCD": "PHASE",
                        "TSPARM": "Trial Phase", "TSVAL": "PHASE III"})
        ts_rows.append({"STUDYID": study, "DOMAIN": "TS", "TSPARMCD": "NARMS",
                        "TSPARM": "Number of Arms", "TSVAL": str(len(ARMS))})

    seq_counters = {}

    def nextseq(key):
        seq_counters[key] = seq_counters.get(key, 0) + 1
        return seq_counters[key]

    for study in STUDIES:
        study_start = dt.date(2025, 1, 6)
        site_ids = []
        for ci, country in enumerate(COUNTRIES):
            for s in range(SITES_PER_COUNTRY):
                site_ids.append((f"{1000 + ci*10 + s}", country))

        for subj_n in range(1, N_SUBJECTS_PER_STUDY + 1):
            site_id, country = random.choice(site_ids)
            subjid = f"{subj_n:04d}"
            usubjid = f"{study}-{site_id}-{subjid}"

            # enrollment timing
            enroll_offset = random.randint(0, 300)
            ic_date = study_start + dt.timedelta(days=enroll_offset)   # informed consent
            screen_date = ic_date + dt.timedelta(days=random.randint(0, 3))

            is_screen_fail = random.random() < SCREEN_FAIL_RATE
            arm_name, arm_cd = random.choice(ARMS)

            age = random.randint(18, 85)
            sex = random.choice(SEX)

            # ---- DM ----
            if is_screen_fail:
                actarm, actarmcd = "Screen Failure", "SCRNFAIL"
                rfstdtc = ""
                rfendtc = d(screen_date + dt.timedelta(days=random.randint(1, 5)))
            else:
                actarm, actarmcd = arm_name, arm_cd
                first_dose = screen_date + dt.timedelta(days=14)
                rfstdtc = d(first_dose)
                rfendtc = ""  # set later from disposition

            dm = {
                "STUDYID": study, "DOMAIN": "DM", "USUBJID": usubjid,
                "SUBJID": subjid, "SITEID": site_id, "COUNTRY": country,
                "RFICDTC": d(ic_date), "RFSTDTC": rfstdtc, "RFENDTC": rfendtc,
                "AGE": age, "AGEU": "YEARS", "SEX": sex,
                "RACE": random.choice(RACE), "ETHNIC": random.choice(ETHNIC),
                "ARM": arm_name, "ARMCD": arm_cd,
                "ACTARM": actarm, "ACTARMCD": actarmcd,
            }

            # ---- Medical History (independent of treatment) ----
            for _ in range(random.randint(0, 3)):
                term = random.choice(MH_TERMS)
                mh_rows.append({
                    "STUDYID": study, "DOMAIN": "MH", "USUBJID": usubjid,
                    "MHSEQ": nextseq((usubjid, "MH")), "MHTERM": term,
                    "MHDECOD": term.upper(), "MHCAT": "GENERAL MEDICAL HISTORY",
                    "MHSTDTC": d(ic_date - dt.timedelta(days=random.randint(100, 3000))),
                })

            if is_screen_fail:
                # only screening visit + screening labs/vitals, then disposition
                sched = VISIT_SCHEDULE[:1]
                first_dose = None
                dropout = False
            else:
                first_dose = screen_date + dt.timedelta(days=14)
                dropout = random.random() < DROPOUT_RATE
                if dropout:
                    n_visits = random.randint(2, 6)
                    sched = VISIT_SCHEDULE[:n_visits]
                else:
                    sched = VISIT_SCHEDULE

            protocol_dev = random.random() < PROTOCOL_DEVIATION_RATE
            anchor = first_dose if first_dose else screen_date
            last_visit_date = anchor

            for (visitnum, visit, offset) in sched:
                # Simulate occasional missing visit (gap), except screening/baseline
                if visitnum > 2 and random.random() < 0.06:
                    continue
                vdate = anchor + dt.timedelta(days=offset + random.randint(-2, 2))
                last_visit_date = vdate

                # ---- SV ----
                sv_rows.append({
                    "STUDYID": study, "DOMAIN": "SV", "USUBJID": usubjid,
                    "VISITNUM": visitnum, "VISIT": visit,
                    "SVSTDTC": d(vdate), "SVENDTC": d(vdate),
                    "PROTOCOLDEV": "Y" if (protocol_dev and visitnum == 3) else "N",
                })

                # ---- VS ----
                for code, name, unit, lo, hi in VS_TESTS:
                    vs_rows.append({
                        "STUDYID": study, "DOMAIN": "VS", "USUBJID": usubjid,
                        "VSSEQ": nextseq((usubjid, "VS")),
                        "VSTESTCD": code, "VSTEST": name,
                        "VSORRES": rnd_around(lo, hi, 0.05),
                        "VSORRESU": unit, "VISITNUM": visitnum, "VISIT": visit,
                        "VSDTC": d(vdate),
                    })

                # ---- LB (not at every visit) ----
                if visitnum in (1, 2, 4, 6, 7):
                    for code, name, unit, lo, hi, cat in LAB_TESTS:
                        val = rnd_around(lo, hi, 0.15)
                        if val < lo:
                            nrind = "LOW"
                        elif val > hi:
                            nrind = "HIGH"
                        else:
                            nrind = "NORMAL"
                        lb_rows.append({
                            "STUDYID": study, "DOMAIN": "LB", "USUBJID": usubjid,
                            "LBSEQ": nextseq((usubjid, "LB")),
                            "LBTESTCD": code, "LBTEST": name, "LBCAT": cat,
                            "LBORRES": val, "LBORRESU": unit,
                            "LBORNRLO": lo, "LBORNRHI": hi, "LBNRIND": nrind,
                            "VISITNUM": visitnum, "VISIT": visit, "LBDTC": d(vdate),
                        })

                # ---- EG (baseline, week4, EoT) ----
                if visitnum in (2, 4, 7):
                    for code, name, unit, lo, hi in EG_TESTS:
                        eg_rows.append({
                            "STUDYID": study, "DOMAIN": "EG", "USUBJID": usubjid,
                            "EGSEQ": nextseq((usubjid, "EG")),
                            "EGTESTCD": code, "EGTEST": name,
                            "EGORRES": rnd_around(lo, hi, 0.07), "EGORRESU": unit,
                            "VISITNUM": visitnum, "VISIT": visit, "EGDTC": d(vdate),
                        })

                # ---- EC: study drug at dosing visits ----
                if first_dose and visitnum in (2, 3, 4, 5, 6):
                    ec_rows.append({
                        "STUDYID": study, "DOMAIN": "EC", "USUBJID": usubjid,
                        "ECSEQ": nextseq((usubjid, "EC")),
                        "ECTRT": STUDY_DRUGS[study] if arm_cd != "PBO" else "Placebo",
                        "ECDOSE": 0 if arm_cd == "PBO" else random.choice([50, 100, 150]),
                        "ECDOSU": "mg", "ECROUTE": "ORAL",
                        "ECSTDTC": d(vdate), "ECENDTC": d(vdate),
                        "VISITNUM": visitnum, "VISIT": visit,
                    })

            # ---- CM: concomitant meds ----
            if not is_screen_fail:
                for _ in range(random.randint(0, 3)):
                    start = anchor + dt.timedelta(days=random.randint(-30, 60))
                    cm_rows.append({
                        "STUDYID": study, "DOMAIN": "CM", "USUBJID": usubjid,
                        "CMSEQ": nextseq((usubjid, "CM")),
                        "CMTRT": random.choice(CM_DRUGS),
                        "CMDECOD": "", "CMDOSE": random.choice([100, 200, 400, 500]),
                        "CMDOSU": "mg", "CMROUTE": "ORAL",
                        "CMSTDTC": d(start),
                        "CMENDTC": d(start + dt.timedelta(days=random.randint(1, 30))),
                        "CMINDC": random.choice(["Pain", "Prophylaxis", "Infection"]),
                    })

            # ---- AE ----
            if not is_screen_fail and first_dose:
                n_ae = random.choices([0, 1, 2, 3, 4], weights=[30, 30, 20, 12, 8])[0]
                subj_has_serious = random.random() < SERIOUS_AE_RATE
                for i in range(n_ae):
                    term, soc = random.choice(AE_TERMS)
                    ae_start = first_dose + dt.timedelta(days=random.randint(1, 100))
                    duration = random.randint(1, 20)
                    serious = "Y" if (subj_has_serious and i == 0) else "N"
                    sev = random.choice(["MILD", "MODERATE", "SEVERE"])
                    if serious == "Y":
                        sev = random.choice(["MODERATE", "SEVERE"])
                    ae_rows.append({
                        "STUDYID": study, "DOMAIN": "AE", "USUBJID": usubjid,
                        "AESEQ": nextseq((usubjid, "AE")),
                        "AETERM": term, "AEDECOD": term.upper(), "AESOC": soc,
                        "AESTDTC": d(ae_start),
                        "AEENDTC": d(ae_start + dt.timedelta(days=duration)),
                        "AESEV": sev, "AESER": serious,
                        "AEREL": random.choice(["RELATED", "NOT RELATED", "POSSIBLY RELATED"]),
                        "AEOUT": random.choice(["RECOVERED", "RECOVERING", "NOT RECOVERED"]),
                    })

            # ---- DS (disposition) + close out RFENDTC ----
            if is_screen_fail:
                ds_term, ds_decod, ds_cat = "Screen Failure", "SCREEN FAILURE", "DISPOSITION EVENT"
                ds_date = screen_date + dt.timedelta(days=random.randint(1, 5))
            elif dropout:
                ds_term = random.choice(["Adverse Event", "Withdrawal by Subject",
                                         "Lost to Follow-up", "Protocol Deviation"])
                ds_decod, ds_cat = ds_term.upper(), "DISPOSITION EVENT"
                ds_date = last_visit_date
                dm["RFENDTC"] = d(last_visit_date)
            else:
                ds_term, ds_decod, ds_cat = "Completed", "COMPLETED", "DISPOSITION EVENT"
                ds_date = last_visit_date
                dm["RFENDTC"] = d(last_visit_date)

            ds_rows.append({
                "STUDYID": study, "DOMAIN": "DS", "USUBJID": usubjid,
                "DSSEQ": nextseq((usubjid, "DS")), "DSTERM": ds_term,
                "DSDECOD": ds_decod, "DSCAT": ds_cat, "DSSTDTC": d(ds_date),
            })

            dm_rows.append(dm)

    return {
        "DM": dm_rows, "SV": sv_rows, "VS": vs_rows, "LB": lb_rows, "EG": eg_rows,
        "EC": ec_rows, "CM": cm_rows, "AE": ae_rows, "MH": mh_rows, "DS": ds_rows,
        "TA": ta_rows, "TS": ts_rows,
    }


LAB_TESTS_CAT = {code: cat for code, _, _, _, _, cat in LAB_TESTS}


def write_csv(rows, name):
    path = OUTPUT_DIR / f"{name}.csv"
    if not rows:
        return path
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    return path


def main():
    data = build()

    # CSV feeds
    for name in ["DM", "SV", "VS", "LB", "EG", "EC", "CM", "DS", "TA", "TS"]:
        write_csv(data[name], name)

    # AE as JSON (API-style feed)
    with open(OUTPUT_DIR / "AE.json", "w", encoding="utf-8") as f:
        json.dump(data["AE"], f, indent=2)

    # MH as Excel (vendor file)
    pd.DataFrame(data["MH"]).to_excel(OUTPUT_DIR / "MH.xlsx", index=False)

    # Summary
    print("Generated record counts:")
    total = 0
    for name, rows in data.items():
        print(f"  {name:4s}: {len(rows):>8,}")
        total += len(rows)
    print(f"  {'TOTAL':4s}: {total:>8,}")
    print(f"Subjects (DM): {len(data['DM']):,}")
    print(f"Output dir: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()